In [15]:

from alpaca.data import  StockHistoricalDataClient , StockBarsRequest, TimeFrame, TimeFrameUnit
from dotenv import load_dotenv
import os
from datetime import datetime, timedelta, timezone
import pandas as pd
import numpy as np
load_dotenv()

API_KEY = os.getenv("APCA_API_KEY_ID") 
SECRET_KEY = os.getenv("APCA_API_SECRET_KEY")

stock_client = StockHistoricalDataClient(API_KEY,  SECRET_KEY)

request_params= StockBarsRequest(
       symbol_or_symbols=["NVDA"], 
        timeframe=TimeFrame.Day, 
        start= datetime(2025, 10, 1), 
        end = datetime(2025, 10, 31)
    )

bars = stock_client.get_stock_bars(request_params)

In [ ]:


def get_intraday_return(date, symbol): 
    """
    assuming date = t: 
    returns C_t/ O_t -1 
    being C_t  t's closing price and O_t t's opening price.  
    """

    request_params= StockBarsRequest(
       symbol_or_symbols=["NVDA"], 
        timeframe=TimeFrame.Day, 
        start= date, 
        end = date+ timedelta(days = 1)
    )

    bars = stock_client.get_stock_bars(request_params)
    bars = bars.df

    if symbol in bars.index: 
        
        returns = bars["close"]/ bars["open"]-1
        print(f"intraday returns for {date.day}/{date.month}/{date.year}: {returns.iloc[0]:.5f}")
        return returns
    print(f"market closed on {date.day}/{date.month}/{date.year}")
    


def get_daily_return(date, symbol, use_log = False): 
    """
    assuming date = t
    returns C_t/ C_{t-1} -1
    """


    request_params= StockBarsRequest(
       symbol_or_symbols=["NVDA"], 
        timeframe=TimeFrame.Day, 
        start= date+ timedelta(days= -1), 
        end = date+ timedelta(days= 1)
    )

    bars = stock_client.get_stock_bars(request_params)
    bars = bars.df

    if symbol in bars.index and len(bars)>=2: 

        if not use_log: 
            returns = bars.iloc[1]["close"]/ bars.iloc[0]["close"]-1
            print(f"daily returns for {date.day}/{date.month}/{date.year}: {returns:.5f}")
        else: 
            returns = np.log(bars.iloc[1]["close"]/ bars.iloc[0]["close"]) #: log is base e per default.
            print(f"daily log returns for {date.day}/{date.month}/{date.year}: {returns:.5f}")

        return returns
    print(f"market closed on {date.day}/{date.month}/{date.year}")

    



symbol = "NVDA"
date =  datetime(2025, 10, 1)


for i in range (20): 
    date= date+ timedelta(days = 1)
    get_intraday_return(date, symbol)
    get_daily_return(date, symbol)
    get_daily_return(date, symbol, True)



intraday returns for 2/10/2025: -0.00374
daily returns for 2/10/2025: 0.00881
daily log returns for 2/10/2025: 0.00877
intraday returns for 3/10/2025: -0.00830
daily returns for 3/10/2025: -0.00672
daily log returns for 3/10/2025: -0.00675
market closed on 4/10/2025
market closed on 4/10/2025
market closed on 4/10/2025
market closed on 5/10/2025
market closed on 5/10/2025
market closed on 5/10/2025
intraday returns for 6/10/2025: 0.00022
market closed on 6/10/2025
market closed on 6/10/2025
intraday returns for 7/10/2025: -0.00639
daily returns for 7/10/2025: -0.00269
daily log returns for 7/10/2025: -0.00270
intraday returns for 8/10/2025: 0.01361
daily returns for 8/10/2025: 0.02200
daily log returns for 8/10/2025: 0.02176
intraday returns for 9/10/2025: 0.00179
daily returns for 9/10/2025: 0.01830
daily log returns for 9/10/2025: 0.01813
intraday returns for 10/10/2025: -0.05346
daily returns for 10/10/2025: -0.04887
daily log returns for 10/10/2025: -0.05010
market closed on 11/10/

In [ ]:


def get_monthly_return(year, month, symbol): 
    """
    assuming date = t
    month in [1, ... 12]
    returns C_t/ C_{t-1} -1 
    """
    prev_m = (month-1)%12
    if prev_m == 0: 
        prev_m = 12
        next_m = 2
    next_m = (prev_m+2)%12
    if next_m == 0: 
        next_m +=1
    year_s = year 
    if prev_m> next_m: 
        year_s-=1
    request_params= StockBarsRequest(
       symbol_or_symbols=["NVDA"], 
        timeframe=TimeFrame.Month, 
        start= datetime(year_s,prev_m, 15), 
        end=datetime(year, next_m , 15)
    )
    bars = stock_client.get_stock_bars(request_params)
    bars = bars.df
    if symbol in bars.index: 
        returns = bars.iloc[1]["close"]/ bars.iloc[0]["close"]-1
        print(f"monthly returns for period {prev_m}-{month}: {returns:.5f}")
        return returns
    else: 

        return False



symbol = "NVDA"

for i in range(0, 20): 
    get_monthly_return(2023, i%12+1, symbol)
        
    




monthly returns for period 12-1: 0.18831
monthly returns for period 1-2: 0.19646
monthly returns for period 2-3: -0.00101
monthly returns for period 3-4: 0.36344
monthly returns for period 4-5: 0.11809
monthly returns for period 5-6: 0.10465
monthly returns for period 6-7: 0.05620
monthly returns for period 7-8: -0.11865
monthly returns for period 8-9: -0.06251
monthly returns for period 9-10: 0.14689
monthly returns for period 10-11: -0.13644
monthly returns for period 11-12: 0.33687
monthly returns for period 12-1: 0.18831
monthly returns for period 1-2: 0.19646
monthly returns for period 2-3: -0.00101
monthly returns for period 3-4: 0.36344
monthly returns for period 4-5: 0.11809
monthly returns for period 5-6: 0.10465
monthly returns for period 6-7: 0.05620
monthly returns for period 7-8: -0.11865


In [25]:

def get_daily_return_from_df(df): 

    df["day_returns"] = df["close"]/ df["close"].shift(1)-1
    return df["day_returns"]



def get_annual_volatility(start_day, start_month, year): 


    request_params= StockBarsRequest(
       symbol_or_symbols=["NVDA"], 
        timeframe=TimeFrame.Day, 
        start= datetime(year, start_month,start_day ), 
        end= datetime(year, start_month,start_day) + timedelta(days = 365)
    )
    #: observe that this should return around 252 rows, as there are 252 trading days in a year!!

    bars = stock_client.get_stock_bars(request_params)
    bars = bars.df

    returns = get_daily_return_from_df(bars)

    return returns.std()*np.sqrt(252)





print(get_annual_volatility(1,1, 2023))



0.48458137191425504
